# Reaktoro Diagram Types Tutorial

This notebook demonstrates all seven diagram classes provided by `reaktoro.extensions.diagrams`.
Each section uses the built-in **SUPCRTBL** thermodynamic database and a different geochemical
system to showcase one diagram type.

| Section | Diagram class | System | Sweep method |
|---------|--------------|--------|--------------|
| 1 | `SpeciationPlot` | Carbonate speciation vs pH | `sweepPH` |
| 2 | `PredominancePlot` | Iron Pourbaix (Eh–pH) | `sweepPHEhGrid` |
| 3 | `SolubilityPlot` | Calcite Ca solubility (T–P) | `sweepTPGrid` |
| 4 | `ActivityDiagram` | Calcite stability in Ca–CO₃ space | `sweepLgActivityGrid` |
| 5 | `MosaicPlot` | Fe minerals + Fe aqueous overlaid | `sweepPHEhGrid` |
| 6 | `LogfO2pHDiagram` | Fe oxide stability vs log *f*(O₂)–pH | `sweepLogfO2pHGrid` |
| 7 | `TPDiagram` | Calcite/Aragonite polymorphism | `sweepTPGrid` |

> **Requirements**: A local Reaktoro build that includes `EquilibriumSweepSolver`
> (add `build/Reaktoro/Release` or `build/python/package/build/lib` to `sys.path`),
> or an installed version that ships the sweep solver.

## Setup: imports and database

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Locate repository root and try to import from the local build first.
# ---------------------------------------------------------------------------
# In a notebook __file__ is not defined; we use the known repository path.
REPO_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")  # Tutorial/ -> DEW_Experimental_Benchmark/ -> repo root
)

# Try the two most common local-build layouts.
for _candidate in [
    os.path.join(REPO_ROOT, "build", "Reaktoro", "Release"),
    os.path.join(REPO_ROOT, "build", "python", "package", "build", "lib"),
]:
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

try:
    import autodiff  # shipped alongside the pyd in some build layouts
except ImportError:
    pass

try:
    from reaktoro4py import *
    print("Using local Reaktoro build.")
except ModuleNotFoundError:
    from reaktoro import *
    print("Using installed 'reaktoro' package.")

# ---------------------------------------------------------------------------
# Make the diagrams module importable from the local python/package tree.
# ---------------------------------------------------------------------------
_diagrams_path = os.path.join(REPO_ROOT, "python", "package")
if os.path.isdir(_diagrams_path) and _diagrams_path not in sys.path:
    sys.path.insert(0, _diagrams_path)

from reaktoro.extensions.diagrams import (
    SpeciationPlot,
    PredominancePlot,
    SolubilityPlot,
    ActivityDiagram,
    MosaicPlot,
    LogfO2pHDiagram,
    TPDiagram,
    water_lines,
    saturation_curve,
)

print("Diagram classes imported successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Shared database (SUPCRTBL built-in)
# ---------------------------------------------------------------------------
db = SupcrtDatabase("supcrtbl")

# Convenience: print all species names containing a given element (useful for
# exploring the database).
def list_species(element_symbol, aggregate_states=None):
    """Return species names in *db* that contain *element_symbol*."""
    results = []
    for sp in db.species():
        syms = [pair[0].symbol() for pair in sp.elements()]
        if element_symbol in syms:
            if aggregate_states is None or sp.aggregateState() in aggregate_states:
                results.append(sp.name())
    return sorted(results)

print("Aqueous Fe species:", list_species("Fe", [AggregateState.Aqueous]))
print("Fe minerals:", [s for s in list_species("Fe", [AggregateState.Solid])
                        if s in ("Hematite", "Magnetite", "Goethite", "Iron", "Pyrite", "Siderite")])

---
## Section 1 — `SpeciationPlot`: Carbonate speciation vs pH

This reproduces the classic carbonate-system diagram from aqueous geochemistry:
how the activities of CO₂(aq), HCO₃⁻, and CO₃²⁻ change as pH sweeps from 2 to 12
at 25 °C and 1 bar.

Analogous to **CHNOSZ** `ionize.R` / `diagram(type="loga")` calls.

**Sweep method**: `sweepPH` — iterates pH values while holding T, P, and total
dissolved carbon fixed.

In [ ]:
# ---------------------------------------------------------------------------
# 1.1  Chemical system: CO2-H2O at 25 °C, 1 bar
# ---------------------------------------------------------------------------
carbonate_aqueous_species = [
    "H2O(l)", "H+", "OH-",
    "CO2(aq)", "HCO3-", "CO3-2",
]

system_carb = ChemicalSystem(db,
    AqueousPhase(speciate(carbonate_aqueous_species))
)

# ---------------------------------------------------------------------------
# 1.2  EquilibriumSpecs: fix T, P, pH; leave aqueous amounts free
# ---------------------------------------------------------------------------
specs_ph = EquilibriumSpecs(system_carb)
specs_ph.temperature()
specs_ph.pressure()
specs_ph.pH()

# ---------------------------------------------------------------------------
# 1.3  Initial state: 1 kg water + 10 mM total dissolved carbon
# ---------------------------------------------------------------------------
state_carb = ChemicalState(system_carb)
state_carb.temperature(25.0, "celsius")
state_carb.pressure(1.0, "bar")
state_carb.set("H2O(l)", 1.0, "kg")
state_carb.set("HCO3-", 0.01, "mol")  # 10 mM initial total C

# ---------------------------------------------------------------------------
# 1.4  Sweep pH 2 → 12
# ---------------------------------------------------------------------------
solver_ph = EquilibriumSweepSolver(specs_ph)

pH_values = np.linspace(2.0, 12.0, 200)
result_carb = solver_ph.sweepPH(
    state_carb,
    pH_values,
    temperature=25.0, temperature_unit="celsius",
    pressure=1.0,    pressure_unit="bar",
)

# ---------------------------------------------------------------------------
# 1.5  Build SpeciationPlot from sweep result
# ---------------------------------------------------------------------------
sp = SpeciationPlot.from_sweep_result(
    result_carb,
    species=["CO2(aq)", "HCO3-", "CO3-2"],
    xlabel="pH",
    xvalues=pH_values,
    palette="tab10",
)

fig1, ax1 = sp.plot(figsize=(7, 4), ylim=(-14, 0))
ax1.set_title("Carbonate speciation at 25 °C, 1 bar (10 mM total C)", fontsize=11)
plt.tight_layout()
plt.savefig("Section1_SpeciationPlot_carbonate.png", dpi=150)
plt.show()

---
## Section 2 — `PredominancePlot`: Iron Pourbaix (Eh–pH) diagram

The classic Pourbaix diagram for the **Fe–O–H** system at 25 °C, 1 bar.
Stability fields are drawn for Fe²⁺, Fe³⁺, FeOH⁺, FeOH²⁺, HFeO₂⁻, Hematite,
Magnetite, Goethite, and metallic Iron.

Analogous to **CHNOSZ** `Pourbaix.R`.

**Sweep method**: `sweepPHEhGrid` — full pH × Eh grid.

In [ ]:
# ---------------------------------------------------------------------------
# 2.1  Chemical system: Fe-O-H at 25 °C, 1 bar
# ---------------------------------------------------------------------------
fe_aq_species = [
    "H2O(l)", "H+", "OH-",
    "Fe+2", "Fe+3", "FeOH+", "FeOH+2", "HFeO2(aq)", "FeO2-",
]
fe_minerals = ["Hematite", "Magnetite", "Goethite", "Iron"]

system_fe = ChemicalSystem(db,
    AqueousPhase(speciate(fe_aq_species)),
    MineralPhases(*fe_minerals),
)

# ---------------------------------------------------------------------------
# 2.2  EquilibriumSpecs: fix T, P, pH, Eh
# ---------------------------------------------------------------------------
specs_pourbaix = EquilibriumSpecs(system_fe)
specs_pourbaix.temperature()
specs_pourbaix.pressure()
specs_pourbaix.pH()
specs_pourbaix.Eh()

# ---------------------------------------------------------------------------
# 2.3  Initial state: 1 kg water + 1 mM total Fe
# ---------------------------------------------------------------------------
state_fe = ChemicalState(system_fe)
state_fe.temperature(25.0, "celsius")
state_fe.pressure(1.0, "bar")
state_fe.set("H2O(l)", 1.0, "kg")
state_fe.set("Fe+2", 1e-3, "mol")  # 1 mM total Fe

# ---------------------------------------------------------------------------
# 2.4  Sweep pH 0 → 14, Eh -1.0 → 1.2 V
# ---------------------------------------------------------------------------
solver_pourbaix = EquilibriumSweepSolver(specs_pourbaix)

pH_grid  = np.linspace(0.0, 14.0, 100)
Eh_grid  = np.linspace(-1.0,  1.2, 100)

grid_fe = solver_pourbaix.sweepPHEhGrid(
    state_fe, pH_grid, Eh_grid, "V",
    temperature=25.0, temperature_unit="celsius",
    pressure=1.0,    pressure_unit="bar",
)

# ---------------------------------------------------------------------------
# 2.5  Build PredominancePlot
# ---------------------------------------------------------------------------
pourbaix_species = ["Fe+2", "Fe+3", "FeOH+", "FeOH+2", "HFeO2(aq)", "FeO2-",
                    "Hematite", "Magnetite", "Goethite", "Iron"]

pp = PredominancePlot.from_grid_result(grid_fe, pourbaix_species)

fig2, ax2 = pp.plot(figsize=(8, 5), palette="Set3")
pp.add_water_lines(ax2, T_K=298.15, color="navy", linestyle="--", linewidth=1.2)
ax2.set_title("Fe–O–H Pourbaix diagram at 25 °C, 1 bar", fontsize=11)
plt.tight_layout()
plt.savefig("Section2_PredominancePlot_Fe_Pourbaix.png", dpi=150)
plt.show()

---
## Section 3 — `SolubilityPlot`: Calcite Ca solubility vs T and P

Shows the **retrograde solubility** of calcite: dissolved Ca (as log₁₀ molality)
across a temperature–pressure grid from 25 to 300 °C and 1 bar to 1 kbar.
Iso-solubility contours are overlaid at selected values.

Analogous to **CHNOSZ** `minsol.R` but on a T–P rather than T–pH axis pair.

**Sweep method**: `sweepTPGrid`.

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Chemical system: Ca-C-O-H
# ---------------------------------------------------------------------------
calcite_aq_species = [
    "H2O(l)", "H+", "OH-",
    "Ca+2", "CaOH+", "Ca(HCO3)+", "CaCO3(aq)",
    "CO2(aq)", "HCO3-", "CO3-2",
]

system_calc = ChemicalSystem(db,
    AqueousPhase(speciate(calcite_aq_species)),
    MineralPhases("Calcite"),
)

# ---------------------------------------------------------------------------
# 3.2  EquilibriumSpecs: fix T and P (let all amounts adjust)
# ---------------------------------------------------------------------------
specs_tp = EquilibriumSpecs(system_calc)
specs_tp.temperature()
specs_tp.pressure()

# ---------------------------------------------------------------------------
# 3.3  Initial state: 1 kg water + excess Calcite
# ---------------------------------------------------------------------------
state_calc = ChemicalState(system_calc)
state_calc.temperature(25.0, "celsius")
state_calc.pressure(1.0, "bar")
state_calc.set("H2O(l)", 1.0, "kg")
state_calc.set("Calcite", 10.0, "mol")  # excess mineral

# ---------------------------------------------------------------------------
# 3.4  Sweep T: 25 → 300 °C;  P: 1 bar → 1000 bar
# ---------------------------------------------------------------------------
solver_tp = EquilibriumSweepSolver(specs_tp)

T_vals = np.linspace(25.0,  300.0, 80)   # °C
P_vals = np.linspace(1.0,  1000.0, 80)   # bar

grid_calc_tp = solver_tp.sweepTPGrid(
    state_calc,
    T_vals, "celsius",
    P_vals, "bar",
)

# ---------------------------------------------------------------------------
# 3.5  Build SolubilityPlot
# ---------------------------------------------------------------------------
sol = SolubilityPlot.from_grid_result(
    grid_calc_tp, element="Ca",
    xlabel="T (°C)", ylabel="P (bar)",
)

fig3, ax3 = sol.plot(
    figsize=(7, 5),
    levels=25,
    iso_levels=[-3.0, -2.5, -2.0, -1.5, -1.0],
    cmap="plasma_r",
)
ax3.set_title("Calcite solubility: log₁₀ [Ca] (mol/kg), 25–300 °C, 1–1000 bar", fontsize=10)
plt.tight_layout()
plt.savefig("Section3_SolubilityPlot_calcite_TP.png", dpi=150)
plt.show()

---
## Section 4 — `ActivityDiagram`: Calcite stability in log *a*(Ca²⁺) – log *a*(CO₃²⁻) space

Shows where calcite is the stable phase as a function of Ca²⁺ and CO₃²⁻
activities at 25 °C, 1 bar.  The mineral stability field is filled;
aqueous (under-saturated) regions are labelled.
A saturation-index contour at SI = 0 is drawn with `saturation_curve`.

Analogous to **CHNOSZ** `saturation.R`.

**Sweep method**: `sweepLgActivityGrid`.

In [ ]:
# ---------------------------------------------------------------------------
# 4.1  Same Ca-C-O-H system as Section 3 (reuse system_calc)
# ---------------------------------------------------------------------------
specs_act = EquilibriumSpecs(system_calc)
specs_act.temperature()
specs_act.pressure()
specs_act.lgActivity("Ca+2")
specs_act.lgActivity("CO3-2")

# ---------------------------------------------------------------------------
# 4.2  Initial state
# ---------------------------------------------------------------------------
state_act = ChemicalState(system_calc)
state_act.temperature(25.0, "celsius")
state_act.pressure(1.0, "bar")
state_act.set("H2O(l)", 1.0, "kg")
state_act.set("Ca+2",  1e-4, "mol")
state_act.set("CO3-2", 1e-4, "mol")

# ---------------------------------------------------------------------------
# 4.3  Sweep log a(Ca+2): -6 → 0;  log a(CO3-2): -6 → 0
# ---------------------------------------------------------------------------
solver_act = EquilibriumSweepSolver(specs_act)

lga_ca   = np.linspace(-6.0, 0.0, 80)  # log10 activity of Ca+2
lga_co3  = np.linspace(-6.0, 0.0, 80)  # log10 activity of CO3-2

grid_act = solver_act.sweepLgActivityGrid(
    state_act,
    "Ca+2",  lga_ca,
    "CO3-2", lga_co3,
    temperature=25.0, temperature_unit="celsius",
    pressure=1.0,    pressure_unit="bar",
)

# ---------------------------------------------------------------------------
# 4.4  Build ActivityDiagram
# ---------------------------------------------------------------------------
stability_species = ["Ca+2", "Calcite"]

ad = ActivityDiagram.from_grid_result(
    grid_act,
    stability_species,
    xlabel="Ca+2",
    ylabel="CO3-2",
)

fig4, ax4 = ad.plot(figsize=(6, 5), palette="Pastel1")

# Overlay the calcite saturation index = 0 contour
si_calcite = np.asarray(grid_act.saturationIndexGrid("Calcite"), dtype=float)
saturation_curve(ax4, lga_ca, lga_co3, si_calcite,
                 label="Calcite SI = 0",
                 colors="darkred", linewidths=1.5)

ax4.set_title("Calcite stability at 25 °C, 1 bar", fontsize=11)
plt.tight_layout()
plt.savefig("Section4_ActivityDiagram_calcite.png", dpi=150)
plt.show()

---
## Section 5 — `MosaicPlot`: Fe minerals and Fe aqueous species overlaid

A two-layer diagram: mineral stability fields (Hematite, Magnetite, Goethite, Iron)
drawn first, then aqueous Fe species (Fe²⁺, Fe³⁺, FeOH⁺, HFeO₂⁻) overlaid
with 65 % transparency.  The mineral fields provide geological context while
the ion fields show the dominant dissolved species.

Analogous to **CHNOSZ** `mosaic.R`.

**Reuses** `grid_fe` from Section 2.

In [ ]:
# grid_fe was computed in Section 2; reuse it here.

# ---------------------------------------------------------------------------
# 5.1  Separate predominance grids for minerals and aqueous species
# ---------------------------------------------------------------------------
mineral_layer_species = ["Hematite", "Magnetite", "Goethite", "Iron"]
aqueous_layer_species  = ["Fe+2", "Fe+3", "FeOH+", "HFeO2(aq)", "FeO2-"]

min_predominance = grid_fe.predominantSpeciesGrid(mineral_layer_species)
aq_predominance  = grid_fe.predominantSpeciesGrid(aqueous_layer_species)

# ---------------------------------------------------------------------------
# 5.2  Build MosaicPlot with two layers
# ---------------------------------------------------------------------------
layers = [
    {
        "species":      mineral_layer_species,
        "predominance": min_predominance,
        "palette":      "Pastel1",
        "alpha":        1.0,
    },
    {
        "species":      aqueous_layer_species,
        "predominance": aq_predominance,
        "palette":      "tab10",
        "alpha":        0.60,
    },
]

mp = MosaicPlot(pH_grid, Eh_grid, layers, xlabel="pH", ylabel="Eh")

fig5, ax5 = mp.plot(figsize=(8, 5))
mp.add_water_lines(ax5, T_K=298.15, color="navy", linestyle="--", linewidth=1.2)
ax5.set_title("Fe–O–H mosaic diagram at 25 °C, 1 bar", fontsize=11)
plt.tight_layout()
plt.savefig("Section5_MosaicPlot_Fe.png", dpi=150)
plt.show()

---
## Section 6 — `LogfO2pHDiagram`: Iron oxide stability vs log *f*(O₂) and pH

Stability fields for Hematite, Magnetite, Iron, and aqueous Fe species as a
function of oxygen fugacity and pH at 25 °C, 1 bar.  This is the standard
diagram used in igneous and metamorphic petrology to discuss redox buffers
(FMQ, NNO, etc.).

Analogous to **CHNOSZ** `contour.R` / `gold.R`.

**Sweep method**: `sweepLogfO2pHGrid` — constrains O₂ fugacity and pH
simultaneously.  Requires `GaseousPhase("O2(g)")` in the system and
`specs.fugacity("O2")` declared.

> **Common redox buffer log *f*(O₂) values at 25 °C, 1 bar**:
> HM ≈ −70.6, FMQ ≈ −85, IW ≈ −83.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1  Chemical system: Fe-O-H + O2 gas
# ---------------------------------------------------------------------------
fe_aq_species_fo2 = [
    "H2O(l)", "H+", "OH-",
    "Fe+2", "Fe+3", "FeOH+", "FeOH+2", "HFeO2(aq)", "FeO2-",
]

system_fo2 = ChemicalSystem(db,
    AqueousPhase(speciate(fe_aq_species_fo2)),
    GaseousPhase("O2(g)"),  # required for fugacity constraint
    MineralPhases("Hematite", "Magnetite", "Goethite", "Iron"),
)

# ---------------------------------------------------------------------------
# 6.2  EquilibriumSpecs: fix T, P, fugacity(O2), pH
# ---------------------------------------------------------------------------
specs_fo2 = EquilibriumSpecs(system_fo2)
specs_fo2.temperature()
specs_fo2.pressure()
specs_fo2.fugacity("O2")
specs_fo2.pH()

# ---------------------------------------------------------------------------
# 6.3  Initial state
# ---------------------------------------------------------------------------
state_fo2 = ChemicalState(system_fo2)
state_fo2.temperature(25.0, "celsius")
state_fo2.pressure(1.0, "bar")
state_fo2.set("H2O(l)", 1.0, "kg")
state_fo2.set("Fe+2", 1e-3, "mol")
state_fo2.set("O2(g)", 1e-6, "mol")  # small seed for gas phase

# ---------------------------------------------------------------------------
# 6.4  Sweep log fO2: -80 → 0;  pH: 0 → 14
# ---------------------------------------------------------------------------
solver_fo2 = EquilibriumSweepSolver(specs_fo2)

logfO2_vals = np.linspace(-80.0,  0.0, 100)  # log10(fO2/bar)
pH_fo2_vals = np.linspace(  0.0, 14.0, 100)

grid_fo2 = solver_fo2.sweepLogfO2pHGrid(
    state_fo2,
    logfO2_vals, "bar",
    pH_fo2_vals,
    temperature=25.0, temperature_unit="celsius",
    pressure=1.0,    pressure_unit="bar",
)

# ---------------------------------------------------------------------------
# 6.5  Build LogfO2pHDiagram
# ---------------------------------------------------------------------------
fo2_species = ["Fe+2", "Fe+3", "FeOH+", "FeOH+2", "HFeO2(aq)", "FeO2-",
               "Hematite", "Magnetite", "Goethite", "Iron"]

fd = LogfO2pHDiagram.from_grid_result(grid_fo2, fo2_species)

fig6, ax6 = fd.plot(figsize=(8, 5), palette="Set3")

# Annotate common redox buffer positions
for label, lfo2 in [("HM", -70.6), ("FMQ", -85.0)]:
    if logfO2_vals[0] <= lfo2 <= logfO2_vals[-1]:
        ax6.axvline(lfo2, color="gray", linestyle=":", linewidth=1.0)
        ax6.text(lfo2 + 0.5, 13.0, label, fontsize=8, color="gray")

ax6.set_title(r"Fe–O–H stability: $\log f_{O_2}$ vs pH at 25 °C, 1 bar", fontsize=11)
plt.tight_layout()
plt.savefig("Section6_LogfO2pHDiagram_Fe.png", dpi=150)
plt.show()

---
## Section 7 — `TPDiagram`: Calcite–Aragonite polymorphism

Shows which CaCO₃ polymorph (Calcite or Aragonite) is thermodynamically stable
across a temperature–pressure grid from 25 to 500 °C and 1 to 5000 bar.
The Calcite → Aragonite transition boundary is a classical high-pressure
metamorphic reaction (appearing at roughly 4–6 kbar).

**Sweep method**: `sweepTPGrid`.

> Compare with **CHNOSZ** phase stability diagrams and classical SUPCRT
> P–T diagrams (e.g., Carlson 1983).

In [ ]:
# ---------------------------------------------------------------------------
# 7.1  Chemical system: CaCO3 polymorphs only
# The aqueous phase acts as a catalyst medium; minerals compete.
# ---------------------------------------------------------------------------
system_poly = ChemicalSystem(db,
    AqueousPhase("H2O(l) H+ OH- Ca+2 HCO3- CO3-2 CO2(aq)"),
    MineralPhases("Calcite", "Aragonite"),
)

# ---------------------------------------------------------------------------
# 7.2  EquilibriumSpecs: fix T and P
# ---------------------------------------------------------------------------
specs_poly = EquilibriumSpecs(system_poly)
specs_poly.temperature()
specs_poly.pressure()

# ---------------------------------------------------------------------------
# 7.3  Initial state: minimal water + excess of both minerals
# ---------------------------------------------------------------------------
state_poly = ChemicalState(system_poly)
state_poly.temperature(25.0, "celsius")
state_poly.pressure(1.0, "bar")
state_poly.set("H2O(l)", 1.0, "kg")
state_poly.set("Calcite",  5.0, "mol")  # both present; equilibrium decides
state_poly.set("Aragonite", 5.0, "mol")

# ---------------------------------------------------------------------------
# 7.4  Sweep T: 25 → 500 °C;  P: 1 bar → 5000 bar
# ---------------------------------------------------------------------------
solver_poly = EquilibriumSweepSolver(specs_poly)

T_poly = np.linspace(25.0,  500.0, 100)  # °C
P_poly = np.linspace(1.0,  5000.0, 100)  # bar

grid_poly = solver_poly.sweepTPGrid(
    state_poly,
    T_poly, "celsius",
    P_poly, "bar",
)

# ---------------------------------------------------------------------------
# 7.5  Build TPDiagram
# ---------------------------------------------------------------------------
poly_species = ["Calcite", "Aragonite"]

td = TPDiagram.from_grid_result(
    grid_poly,
    poly_species,
    T_unit="C",
    P_unit="bar",
)

fig7, ax7 = td.plot(figsize=(7, 5), palette="Set1")
ax7.set_title("CaCO₃ polymorph stability (Calcite vs Aragonite)", fontsize=11)
ax7.set_xlabel("T (°C)")
ax7.set_ylabel("P (bar)")
plt.tight_layout()
plt.savefig("Section7_TPDiagram_CaCO3_polymorphs.png", dpi=150)
plt.show()

---
## Summary

| Section | Diagram class | Key result |
|---------|--------------|------------|
| 1 | `SpeciationPlot` | CO₂/HCO₃⁻/CO₃²⁻ cross-over at pH 6.3 and 10.3 |
| 2 | `PredominancePlot` | Fe Pourbaix: hematite/magnetite/iron metal fields |
| 3 | `SolubilityPlot` | Calcite retrograde solubility with T; complex P dependence |
| 4 | `ActivityDiagram` | Calcite stability boundary in log *a*(Ca²⁺) vs log *a*(CO₃²⁻) |
| 5 | `MosaicPlot` | Mineral + aqueous Fe fields simultaneously |
| 6 | `LogfO2pHDiagram` | Fe redox transitions at geologically relevant fO₂ |
| 7 | `TPDiagram` | Calcite → Aragonite transition near 4–6 kbar |

All calculations use `EquilibriumSweepSolver` with the built-in SUPCRTBL database.
To adapt these examples to other systems, change the species lists and sweep ranges;
use `list_species()` (defined in the Setup cell) to explore what is available.